# Experimentos — Optuna + MLflow Registry

Optimización de hiperparámetros con **Optuna**, runs anidados en MLflow, modelo final y registro en **Model Registry**.

**Importante:** `optimize_hyperparameters` y `train_model` deben ejecutarse dentro del **mismo** run padre de MLflow (igual que `training_flow.py`), para que métricas y jerarquía de runs sean coherentes.

In [ ]:
from __future__ import annotations

import os
import sys
from pathlib import Path

def _find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "healthcare_fraud").exists():
            return candidate
    return start

_root = _find_repo_root(Path.cwd().resolve())
os.environ["DATA_DIR"] = str((_root / "data").resolve())
_src = _root / "src"
if _src.is_dir():
    sp = str(_src)
    if sp not in sys.path:
        sys.path.insert(0, sp)

import warnings

import mlflow
import pandas as pd

from healthcare_fraud.data import clean_dataframe, load_dataset, validate_dataframe
from healthcare_fraud.features import (
    FEATURE_COLS,
    build_features,
    prepare_train_val,
    split_providers,
)
from healthcare_fraud.models import (
    optimize_hyperparameters,
    register_model,
    setup_mlflow,
    train_model,
    transition_model_stage,
)

warnings.filterwarnings("ignore")
print("Imports OK")

## 1. Datos

In [ ]:
raw_tables = load_dataset()
clean_tables = {
    name: clean_dataframe(validate_dataframe(df, name), name) for name, df in raw_tables.items()
}
print("Tablas limpias:", {k: v.shape for k, v in clean_tables.items()})

In [ ]:
feature_df = build_features(clean_tables)
train_df, val_df = split_providers(feature_df)

X_train, X_val, y_train, y_val, preprocessor = prepare_train_val(train_df, val_df)
X_train_raw = train_df[FEATURE_COLS].values
X_val_raw = val_df[FEATURE_COLS].values

n_fraud = int(y_train.sum())
n_total = len(y_train)
print(f"Train: {n_total} providers | Fraude: {n_fraud} ({n_fraud / n_total:.1%})")

## 2–3. Optuna + modelo final (un solo run padre)

In [ ]:
setup_mlflow()

with mlflow.start_run(run_name="optuna_experiment") as parent_run:
    mlflow.set_tag("notebook", "03_experiments")
    best_params = optimize_hyperparameters(X_train, y_train, X_val, y_val)
    run_id, metrics = train_model(
        X_train_raw, y_train, X_val_raw, y_val, preprocessor, best_params
    )

print("Mejores hiperparámetros:")
print(pd.Series(best_params).to_string())
print(f"\nRun ID (modelo final): {run_id}")
pd.DataFrame([metrics], index=["optimizado"]).round(4)

## 4. Model Registry

In [ ]:
MODEL_NAME = "healthcare-fraud-detector"

mv = register_model(run_id, MODEL_NAME)
print(f"Modelo registrado: {mv.name} v{mv.version}")

In [ ]:
mv_staging = transition_model_stage(MODEL_NAME, mv.version, "Staging")
print(f"Stage: {mv_staging.current_stage}")

## Conclusiones

- Ver resultados: `uv run mlflow ui --port 5001 --backend-store-uri sqlite:///mlflow.db`
- Modelo en registry (p. ej. **Staging**).
- Orquestación end-to-end: `04_prefect_flow.ipynb` o `healthcare_fraud.pipelines.training_flow`.